In [1]:
!pip uninstall -y transformers
!pip install -U transformers accelerate bitsandbytes sentencepiece

Found existing installation: transformers 5.12.0
Uninstalling transformers-5.12.0:
  Successfully uninstalled transformers-5.12.0
  Using cached transformers-5.12.0-py3-none-any.whl.metadata (33 kB)
Using cached transformers-5.12.0-py3-none-any.whl (11.2 MB)


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import pandas as pd
import re

In [3]:
def extract_final_num(text):
    match = re.search(r"Final\s+Answer:\s*(-?\d+\.?\d*)", text, re.IGNORECASE)
    if match:
        return match.group(1).strip('.')
    nums = re.findall(r'-?\d+\.?\d*', text)
    if nums:
        return nums[-1].strip('.')
    return None

In [4]:
test_df = pd.read_csv("unified_svamp_test.csv").head(50)

In [5]:
import os
os.environ['LD_LIBRARY_PATH'] += ':/usr/local/cuda-13.0/lib64'

In [6]:
from google.colab import userdata
from huggingface_hub import login

In [9]:
hf_token = userdata.get('hf_token')
login(hf_token)
print("Authenticated successfully!")

Authenticated successfully!


In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
model_id = "microsoft/Phi-3-mini-4k-instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type = "nf4",
    bnb_4bit_compute_dtype=torch.float16
)

In [11]:
tokenizer = AutoTokenizer.from_pretrained(model_id, add_bos_token=False)
stop_tokens = ["<|end|>", "<|endoftext|>"]
eos_token_ids = [tokenizer.convert_tokens_to_ids(t) for t in stop_tokens if tokenizer.convert_tokens_to_ids(t) is not None]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

In [12]:
print("Downloading the model")
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")
print("Sucess! The engine is running")

model.safetensors.index.json:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Sucess! The engine is running


In [13]:
results = []
correct_count = 0
total_count = len(test_df)

In [14]:
print("\nStarting Baseline Evaluation-->")
for index, row in test_df.iterrows():
    question = row['question']
    ground_truth = str(row['answer']).strip()
    messages = [
    {"role": "user", "content": f"You are an expert math tutor. Question: {question}\nAnswer the question step-by-step. You must strictly end your response with 'Final Answer: <number>'."}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.1,
        do_sample=False,
        eos_token_id=eos_token_ids,
    )
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
    extracted_answer = extract_final_num(response)

    is_correct = False
    if extracted_answer is not None:
        try:
            if float(extracted_answer) == float(ground_truth):
                is_correct = True
                correct_count += 1
        except ValueError:
            pass

    print(f"\n--- Problem {index + 1} ---")
    print(f"Question: {question}")
    print(f"Model Output:\n{response.strip()}")
    print(f"Extracted Answer: {extracted_answer}")
    print(f"Ground Truth Answer: {ground_truth}")
    print(f"Correct: {is_correct}")
    print("-" * 30)

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Starting Baseline Evaluation-->


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



--- Problem 1 ---
Question: There are 87 oranges and 290 bananas in Philip's collection. If the bananas are organized into 2 groups and oranges are organized into 93 groups How big is each group of bananas?
Model Output:
Step 1: Determine the total number of bananas.
Philip has 290 bananas in his collection.

Step 2: Divide the total number of bananas by the number of groups.
Philip organizes the bananas into 2 groups.

290 bananas ÷ 2 groups = 145 bananas per group

Final Answer: 145
Extracted Answer: 145
Ground Truth Answer: 145
Correct: True
------------------------------


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blo

...


--- Problem 48 ---
Question: After resting they decided to go for a swim. The depth of the water is 2 times Dean's height. Dean is 8 feet shorter than Ron. If Ron stands at 14 feet How deep was the water?
Model Output:
Step 1: Determine Ron's height.
Ron stands at 14 feet.

Step 2: Determine Dean's height.
Dean is 8 feet shorter than Ron, so we subtract 8 feet from Ron's height:
14 feet - 8 feet = 6 feet.

Step 3: Determine the depth of the water.
The depth of the water is 2 times Dean's height, so we multiply Dean's height by 2:
6 feet * 2 = 12 feet.

Final Answer: 12.
Extracted Answer: 12
Ground Truth Answer: 12
Correct: True
------------------------------


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blo


--- Problem 49 ---
Question: Every day Ryan spends 6 hours on learning english and 7 hours on learning chinese. If he learns for 5 days How many hours does he spend on learning english and chinese in all?
Model Output:
Step 1: Calculate the total hours spent on learning English per day.
Ryan spends 6 hours on learning English every day.

Step 2: Calculate the total hours spent on learning Chinese per day.
Ryan spends 7 hours on learning Chinese every day.

Step 3: Calculate the total hours spent on learning both English and Chinese per day.
Total hours spent per day = Hours spent on English + Hours spent on Chinese
Total hours spent per day = 6 hours (English) + 7 hours (Chinese)
Total hours spent per day = 13 hours

Step 4: Calculate the total hours spent on learning both English and Chinese over 5 days.
Total hours spent over 5 days = Total hours spent per day * Number of days
Total hours spent over 5 days = 13 hours * 5 days
Total hours spent over 5 days = 65 hours

Final Answer: 6

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blo


--- Problem 50 ---
Question: Jack received 10 emails in the morning, 5 emails in the afternoon and 4 emails in the evening. How many more emails did Jack receive in the afternoon than in the evening?
Model Output:
Step 1: Identify the number of emails received in the afternoon.
Jack received 5 emails in the afternoon.

Step 2: Identify the number of emails received in the evening.
Jack received 4 emails in the evening.

Step 3: Calculate the difference between the number of emails received in the afternoon and the evening.
Difference = Emails in the afternoon - Emails in the evening
Difference = 5 - 4

Step 4: Calculate the final answer.
Difference = 1

Final Answer: 1.
Extracted Answer: 1
Ground Truth Answer: 1
Correct: True
------------------------------


In [15]:
accuracy = (correct_count / total_count) * 100
print(f"FINAL BASELINE ACCURACY: {accuracy:.2f}% ({correct_count}/{total_count})")

FINAL BASELINE ACCURACY: 82.00% (41/50)
